In [1]:
import time
import torch
from torch.distributions import Uniform
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sbi.inference import NLE_A, NPE_C, NRE_B
from sbi.utils.user_input_checks import MultipleIndependent
import sys
sys.path.append('../pysimARG')
from discrete_uniform import DiscreteUniform
from clonal_genealogy import ClonalTree
from ClonalOrigin_seq_sim import ClonalOrigin_seq_sim

try:
    from sbi.inference.posteriors import MCMCPosteriorParameters

    MCMC_PARAMS = MCMCPosteriorParameters(
        method="slice_np_vectorized",
        thin=1,
        warmup_steps=100,
        num_chains=4,
        init_strategy="sir",
        init_strategy_parameters={"num_candidate_samples": 1000},
        num_workers=1,
    )
except Exception:
    MCMC_PARAMS = None

DEVICE = "cpu"

c:\Users\u2008181\likelihood-free\sbi_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
np.random.seed(100)
clonal_tree = ClonalTree(n=30)
edge = clonal_tree.edge
node_height = clonal_tree.node_height

In [3]:
edge, node_height

(array([[3.10000000e+01, 1.40000000e+01, 1.80220314e-03],
        [3.10000000e+01, 2.90000000e+01, 1.80220314e-03],
        [3.20000000e+01, 2.80000000e+01, 5.97471972e-03],
        [3.20000000e+01, 2.60000000e+01, 5.97471972e-03],
        [3.30000000e+01, 3.00000000e+00, 8.25024952e-03],
        [3.30000000e+01, 2.70000000e+01, 8.25024952e-03],
        [3.40000000e+01, 2.20000000e+01, 1.30036692e-02],
        [3.40000000e+01, 1.00000000e+01, 1.30036692e-02],
        [3.50000000e+01, 1.10000000e+01, 2.34636374e-02],
        [3.50000000e+01, 1.80000000e+01, 2.34636374e-02],
        [3.60000000e+01, 3.30000000e+01, 1.80941931e-02],
        [3.60000000e+01, 2.10000000e+01, 2.63444427e-02],
        [3.70000000e+01, 1.70000000e+01, 3.56575696e-02],
        [3.70000000e+01, 6.00000000e+00, 3.56575696e-02],
        [3.80000000e+01, 1.20000000e+01, 3.73864931e-02],
        [3.80000000e+01, 3.20000000e+01, 3.14117734e-02],
        [3.90000000e+01, 3.00000000e+01, 3.85383268e-02],
        [3.900

In [4]:
print(clonal_tree.height)
print(clonal_tree.length)

1.3283859639923548
5.896919901171329


In [5]:
theta1 = np.loadtxt('../data/ClonalOrigin/rho_and_theta/theta1.csv', delimiter=",")
x1 = np.loadtxt('../data/ClonalOrigin/rho_and_theta/x1.csv', delimiter=",")

theta2 = np.loadtxt('../data/ClonalOrigin/rho_and_theta/theta2.csv', delimiter=",")
x2 = np.loadtxt('../data/ClonalOrigin/rho_and_theta/x2.csv', delimiter=",")

theta3 = np.loadtxt('../data/ClonalOrigin/rho_and_theta/theta3.csv', delimiter=",")
x3 = np.loadtxt('../data/ClonalOrigin/rho_and_theta/x3.csv', delimiter=",")

theta4 = np.loadtxt('../data/ClonalOrigin/rho_and_theta/theta4.csv', delimiter=",")
x4 = np.loadtxt('../data/ClonalOrigin/rho_and_theta/x4.csv', delimiter=",")

x = np.vstack([x1, x2, x3, x4])
theta = np.vstack([theta1, theta2, theta3, theta4])

print(theta.shape, x.shape)

(50000, 3) (50000, 46)


In [6]:
theta = torch.tensor(theta, device=DEVICE)
theta = theta.to(torch.float32)
theta_numpy = theta.cpu().numpy()

x = torch.tensor(x, device=DEVICE)
x = x.to(torch.float32)
x_numpy = x.cpu().numpy()

In [7]:
theta_test = np.loadtxt('../data/ClonalOrigin/rho_and_theta/theta_sbc.csv', delimiter=",")
x_test = np.loadtxt('../data/ClonalOrigin/rho_and_theta/x_sbc.csv', delimiter=",")

print(theta_test.shape, x_test.shape)
theta_test = torch.tensor(theta_test, device=DEVICE)
theta_test = theta_test.to(torch.float32)
theta_numpy = theta_test.cpu().numpy()

x_test = torch.tensor(x_test, device=DEVICE)
x_test = x_test.to(torch.float32)
x_test_numpy = x_test.cpu().numpy()

(1000, 3) (1000, 46)


In [8]:
BUDGETS = [200, 500, 1000, 2000, 5000, 10000, 20000, 50000]

METHOD_CONFIGS = {
    "nle": {
        "cls": NLE_A,
        "init_kwargs": {
            "density_estimator": "nsf",
        },
        "train_kwargs": {},
    },
    "npe": {
        "cls": NPE_C,
        "init_kwargs": {
            "density_estimator": "nsf",
        },
        "train_kwargs": {},
    },
    "nre": {
        "cls": NRE_B,
        "init_kwargs": {
            "classifier": "resnet",
        },
        "train_kwargs": {},
    },
}

NUM_POSTERIOR_SAMPLES = 1000
NUM_PPC_SAMPLES = 50
TEST_BATCH_SIZE = 50
SIM_BATCH_SIZE = 2048

TRAIN_KWARGS_COMMON = dict(
    training_batch_size=200,
    validation_fraction=0.1,
    stop_after_epochs=20,
    max_num_epochs=300,
    learning_rate=0.0005,
    show_train_summary=False,
)

prior_rho = Uniform(low=torch.tensor([0.0]), high=torch.tensor([0.1]))
prior_theta = Uniform(low=torch.tensor([0.0]), high=torch.tensor([0.1]))
prior_L = DiscreteUniform(low=torch.tensor([100.0]), high=torch.tensor([10000.0]))

prior = MultipleIndependent(
    dists=[prior_rho, prior_theta, prior_L],
    validate_args=False,
    device=DEVICE
)


def to_device_float(t):
    return t.detach().clone().float().to(DEVICE)

In [9]:
def simulator(theta):
    theta = theta.reshape(-1)
    summary_stats = ClonalOrigin_seq_sim(clonal_tree,
                                         theta[0].item(),
                                         theta[1].item(),
                                         int(theta[2].item()),
                                         300)
    summary_stats = torch.tensor(summary_stats, device=DEVICE)
    return summary_stats

In [10]:
theta_all = to_device_float(theta)
x_all = to_device_float(x)
theta_test_all = to_device_float(theta_test)
x_test_all = to_device_float(x_test)

theta_dim = theta_all.shape[1]
x_dim = x_all.shape[1]

# Use a fixed permutation
g = torch.Generator(device="cpu").manual_seed(123)
perm = torch.randperm(theta_all.shape[0], generator=g).to(DEVICE)

theta_all = theta_all[perm]
x_all = x_all[perm]

# Metric standardization constants.
theta_scale = theta_all.std(dim=0).clamp_min(1e-8).cpu()
x_center = x_all.mean(dim=0).cpu()
x_scale = x_all.std(dim=0).clamp_min(1e-8).cpu()

In [11]:
def build_posterior_robust(inference, estimator, method_name, prior):
    if method_name == "npe":
        return inference.build_posterior(
            estimator,
            prior=prior,
        )

    if method_name in {"nle", "nre"}:
        try:
            return inference.build_posterior(
                estimator,
                prior=prior,
                posterior_parameters=MCMC_PARAMS,
            )
        except TypeError:
            return inference.build_posterior(
                estimator,
                prior=prior,
                sample_with="mcmc",
                mcmc_method="slice_np_vectorized",
                mcmc_parameters=dict(
                    num_chains=4,
                    warmup_steps=100,
                    thin=1,
                    init_strategy="sir",
                    init_strategy_parameters={"num_candidate_samples": 1000},
                ),
            )

    raise ValueError(f"Unknown method_name: {method_name}")

In [12]:
def train_sbi_model(method_name, config, theta_train, x_train, prior):
    method_cls = config["cls"]

    inference = method_cls(
        prior=prior,
        device=DEVICE,
        **config["init_kwargs"],
    )

    inference.append_simulations(theta_train, x_train)

    train_kwargs = {
        **TRAIN_KWARGS_COMMON,
        **config["train_kwargs"],
    }

    estimator = inference.train(**train_kwargs)

    posterior = build_posterior_robust(
        inference=inference,
        estimator=estimator,
        method_name=method_name,
        prior=prior,
    )

    return posterior

In [13]:
def sample_posterior_one_by_one(
    posterior,
    x_eval,
    num_samples=1000,
):
    """
    Sample posterior one test observation at a time.

    posterior: trained sbi posterior
    x_eval:    tensor with shape [num_test, x_dim], e.g. [1000, 46]

    returns:
        posterior_samples with shape [num_test, num_samples, theta_dim]
        e.g. [1000, 1000, 3]
    """
    all_samples = []

    for i in tqdm(range(x_eval.shape[0]), desc="Sampling posteriors"):
        x_i = x_eval[i, :]  # shape [46]

        samples_i = posterior.sample(
            (num_samples,),
            x=x_i,
            show_progress_bars=False,
        )

        # Ensure shape [num_samples, theta_dim]
        samples_i = samples_i.reshape(num_samples, -1)

        all_samples.append(samples_i.detach().cpu())

    posterior_samples = torch.stack(all_samples, dim=0)

    return posterior_samples

In [14]:
def compute_posterior_metrics(samples, theta_true):
    """
    samples:    [K, S, D]
    theta_true: [K, D]
    """
    theta_true = theta_true.cpu()

    lower = samples.quantile(0.025, dim=1)
    upper = samples.quantile(0.975, dim=1)

    inside = (theta_true >= lower) & (theta_true <= upper)

    coverage_per_dim = inside.float().mean(dim=0)
    coverage_marginal_mean = coverage_per_dim.mean()

    coverage_box = inside.all(dim=1).float().mean()

    posterior_mean = samples.mean(dim=1)
    err = posterior_mean - theta_true

    mean_abs_error_per_dim = err.abs().mean(dim=0)
    mean_l2_error = err.norm(dim=1).mean()

    standardized_rmse = torch.sqrt(((err / theta_scale) ** 2).mean())

    metrics = {
        "coverage95_marginal_mean": coverage_marginal_mean.item(),
        "coverage95_box_all_dims": coverage_box.item(),
        "posterior_mean_l2_error": mean_l2_error.item(),
        "posterior_mean_standardized_rmse": standardized_rmse.item(),
    }

    for d in range(theta_true.shape[1]):
        metrics[f"coverage95_theta{d}"] = coverage_per_dim[d].item()
        metrics[f"mean_abs_error_theta{d}"] = mean_abs_error_per_dim[d].item()

    return metrics

In [15]:
@torch.no_grad()
def simulate_in_batches(theta_flat, simulator, batch_size=2048):
    xs = []

    for start in range(0, theta_flat.shape[0], batch_size):
        tb = theta_flat[start:start + batch_size]
        xb = simulator(tb.cpu())

        if not torch.is_tensor(xb):
            xb = torch.as_tensor(xb)

        xs.append(xb.detach().float().cpu())

    return torch.cat(xs, dim=0)

In [16]:
def compute_posterior_predictive_discrepancy(
    samples,
    x_obs,
    simulator,
    num_ppc_samples=50,
    sim_batch_size=2048,
):
    """
    Simple posterior predictive discrepancy:

        E_{theta ~ posterior, x_rep ~ simulator(theta)}
        [ standardized_RMSE(x_rep, x_obs) ]

    Lower is better.

    For your 46-dimensional x, this uses identity summaries by default.
    If your x has domain-specific summaries, replace x_rep and x_obs
    with summary_fn(x_rep), summary_fn(x_obs).
    """
    x_obs = x_obs.cpu()

    K, S, D = samples.shape
    P = min(num_ppc_samples, S)

    theta_ppc = samples[:, :P, :].reshape(K * P, D)
    x_rep = simulate_in_batches(
        theta_ppc,
        simulator=simulator,
        batch_size=sim_batch_size,
    )
    x_rep = x_rep.reshape(K, P, -1)

    z_rep = (x_rep - x_center[None, None, :]) / x_scale[None, None, :]
    z_obs = (x_obs[:, None, :] - x_center[None, None, :]) / x_scale[None, None, :]

    rmse_per_rep = torch.sqrt(((z_rep - z_obs) ** 2).mean(dim=-1))

    return {
        "ppc_rmse_mean": rmse_per_rep.mean().item(),
        "ppc_rmse_median": rmse_per_rep.median().item(),
    }

In [17]:
rows = []

for n_sims in BUDGETS:
    theta_train = theta_all[:n_sims]
    x_train = x_all[:n_sims]

    for method_name, config in METHOD_CONFIGS.items():
        print(f"\nTraining {method_name.upper()} with {n_sims} simulations")

        start_time = time.time()

        posterior = train_sbi_model(
            method_name=method_name,
            config=config,
            theta_train=theta_train,
            x_train=x_train,
            prior=prior,
        )

        train_time = time.time() - start_time

        print(f"Sampling posterior for {method_name.upper()}, n={n_sims}")

        sample_start = time.time()

        posterior_samples = sample_posterior_one_by_one(
            posterior=posterior,
            x_eval=x_test_all,
            num_samples=1000,
        )

        sample_time = time.time() - sample_start

        posterior_metrics = compute_posterior_metrics(
            samples=posterior_samples,
            theta_true=theta_test_all,
        )

        ppc_metrics = compute_posterior_predictive_discrepancy(
            samples=posterior_samples,
            x_obs=x_test_all,
            simulator=simulator,
            num_ppc_samples=NUM_PPC_SAMPLES,
            sim_batch_size=SIM_BATCH_SIZE,
        )

        row = {
            "method": method_name,
            "num_simulations": n_sims,
            "train_time_sec": train_time,
            "sample_time_sec": sample_time,
            **posterior_metrics,
            **ppc_metrics,
        }

        rows.append(row)

        results_df = pd.DataFrame(rows)
        results_df.to_csv("sbi_benchmark_results_partial.csv", index=False)

        print(results_df.tail(1).T)


results_df = pd.DataFrame(rows)
results_df.to_csv("sbi_benchmark_results.csv", index=False)

print(results_df)


Training NLE with 200 simulations


c:\Users\u2008181\likelihood-free\sbi_env\Lib\site-packages\sbi\inference\trainers\base.py:296: UserWarning: Data has extreme outliers in dimension(s) [11, 26, 27, 43] (beyond 10.0x IQR from quartiles). This may cause precision loss during z-scoring, where distinct values become indistinguishable. Consider removing outliers from your data or setting `z_score_x='none'` (though this may affect training).
  warn_if_invalid_for_zscoring(x)


 Neural network successfully converged after 232 epochs.Sampling posterior for NLE, n=200


Sampling posteriors:   0%|          | 0/1000 [00:00<?, ?it/s]c:\Users\u2008181\likelihood-free\sbi_env\Lib\site-packages\sbi\inference\posteriors\mcmc_posterior.py:592: UserWarning: As of sbi v0.19.0, the behavior of the SIR initialization for MCMC has changed. If you wish to restore the behavior of sbi v0.18.0, set `init_strategy='resample'.`
  init_fn = self._build_mcmc_init_fn(
Sampling posteriors:   0%|          | 2/1000 [01:58<16:21:30, 59.01s/it]


KeyboardInterrupt: 